In [16]:
from langgraph.graph import StateGraph, START,END
from langchain_ollama import ChatOllama
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field
import operator

In [17]:
model = ChatOllama(model='llama3.2')

In [18]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description='Detailed Feedback for the essay')
    score: int = Field(description='Score out of 10',ge=0,le=10)


In [19]:
structured_model = model.with_structured_output(EvaluationSchema)

In [20]:
essay = """The title “King of Pop” is not merely an honorific; it is a coronation, a recognition of unparalleled influence, and a testament to an artist who fundamentally reshaped the global cultural landscape. That artist is Michael Jackson. More than a singer, dancer, or entertainer, Jackson was a phenomenon—a creative force whose work transcended genre, race, and geography to create a universal language of music and movement. His reign was built on a foundation of prodigious talent, visionary artistry, and an almost mythical persona, leaving a legacy that continues to echo through every corner of popular music.

Jackson’s destiny seemed preordained from his earliest days in Gary, Indiana. As the electrifying lead singer of the Jackson 5, he was a child prodigy whose powerful, soulful voice belied his young age. Hits like “I Want You Back” and “ABC” showcased not just a remarkable vocal instrument but an innate stage presence and charisma that captivated audiences. This Motown era was the crucible that forged his talent, but it was only the prologue. His transition into a solo adult artist began with the albums Off the Wall (1979), a masterful blend of disco, funk, and soul that produced timeless hits like “Don’t Stop ‘Til You Get Enough” and “Rock with You.” It was a massive success, but it was merely a prelude to the seismic event that would follow.

In 1982, Jackson released Thriller, an album that didn't just dominate music but annexed global culture. It became the best-selling album of all time, a record it still holds. But its significance lies beyond numbers. Thriller was a masterpiece of sonic ambition, seamlessly weaving together pop, rock, funk, and R&B. Every track was an event. The album’s success was turbocharged by Jackson’s revolutionary use of the music video. He transformed the format from a promotional tool into a high-concept art form and short film. The videos for “Billie Jean,” “Beat It,” and the nearly 14-minute epic “Thriller” were cultural milestones. They broke racial barriers on MTV, set new standards for production value, and offered a visual spectacle that the world eagerly gathered to watch. His iconic dance moves—the moonwalk, the robotic precision, the effortless spins—became a new vocabulary for performance, studied and imitated by millions.

Jackson’s artistry was defined by meticulous perfectionism and innovation. His sound was instantly recognizable: his signature vocal hiccups, breathy whispers, and soaring cries were emotionally potent and technically stunning. In the studio, he was a relentless taskmaster, working alongside producer Quincy Jones to create dense, layered soundscapes that were both catchy and complex. His songwriting tackled a wide range of themes, from paranoia and social commentary (“Billie Jean,” “Beat It”) to pure escapist joy (“Wanna Be Startin’ Somethin’”) and heartfelt humanitarian pleas (“Heal the World,” “Man in the Mirror”). He used his unprecedented platform to advocate for global issues, making his music a force for unity and awareness.

However, the life of the King was also a Shakespearean tragedy played out under the blinding glare of the modern media. His immense fame became a gilded cage, isolating him from a world that simultaneously adored and scrutinized him. His changing appearance fueled endless speculation, and his unconventional lifestyle made him a constant source of tabloid fascination. Later in his life, serious allegations and legal battles cast a long and dark shadow over his legacy, creating a complex and often contradictory narrative that continues to be debated.

Yet, these controversies cannot eclipse his monumental artistic achievements. Michael Jackson’s influence is inextricably woven into the DNA of contemporary music. From the stagecraft of Beyoncé and Usher to the genre-blending of The Weeknd and Bruno Mars, his blueprint is everywhere. He demonstrated the global power of a pop record and set the benchmark for live performance, from his legendary Motown 25 performance to his massive world tours.

In conclusion, Michael Jackson earned his crown through a combination of divine talent, fearless innovation, and transformative impact. He was a dancer who defied gravity, a singer who channeled raw emotion, and a visionary who saw the future of entertainment. While his personal life remains a subject of complex discussion, his artistic contributions are undeniable and immortal. He didn't just rule the charts; he expanded the very possibilities of pop music, forever cementing his status as its one, true, and undisputed King."""

In [21]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10\n {essay}'
structured_model.invoke(prompt)

EvaluationSchema(feedback="The essay provides a comprehensive analysis of Michael Jackson's impact on popular music, exploring his innovative style, artistic achievements, and enduring influence. The writer effectively uses vivid descriptions and specific examples to illustrate Jackson's creative force, sonic ambition, and genre-bending approach. However, the text could benefit from more nuanced and balanced discussion of controversies surrounding Jackson's personal life, as well as a clearer structure for supporting arguments and claims.", score=8)

In [22]:
structured_model.invoke(prompt).score

9

In [32]:
class EssayState(TypedDict):
    essay:str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_score:Annotated[list[int],operator.add]
    avg_score:float

In [33]:
def evaluate_language(state: EssayState):
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10\n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_score': [output.score]}

In [34]:
def evaluate_analysis(state: EssayState):
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10\n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_score': [output.score]}

In [35]:
def evaluate_thought(state: EssayState):
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10\n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_score': [output.score]}

In [39]:
def final_eval(state: EssayState):
    prompt = f'Based on the following feedbacks, create a summarized feedback \n language Feedback - {state['language_feedback']}\n depth of analysis feedback -{state["analysis_feedback"]}\n clarity of thought feedback - {state['clarity_feedback']}'
    output = model.invoke(prompt).content

    avg_score = sum(state['individual_score']) / len(state['individual_score'])

    return {'overall_feedback': output, 'avg_score': avg_score}

In [40]:
graph = StateGraph(EssayState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_eval',final_eval)

graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_thought')
graph.add_edge('evaluate_thought','final_eval')
graph.add_edge('evaluate_language','final_eval')
graph.add_edge('evaluate_thought','final_eval')
graph.add_edge('final_eval',END)

workflow = graph.compile()

In [41]:
initial_state = {
    'essay':essay
}

workflow.invoke(initial_state)

{'essay': "The title “King of Pop” is not merely an honorific; it is a coronation, a recognition of unparalleled influence, and a testament to an artist who fundamentally reshaped the global cultural landscape. That artist is Michael Jackson. More than a singer, dancer, or entertainer, Jackson was a phenomenon—a creative force whose work transcended genre, race, and geography to create a universal language of music and movement. His reign was built on a foundation of prodigious talent, visionary artistry, and an almost mythical persona, leaving a legacy that continues to echo through every corner of popular music.\n\nJackson’s destiny seemed preordained from his earliest days in Gary, Indiana. As the electrifying lead singer of the Jackson 5, he was a child prodigy whose powerful, soulful voice belied his young age. Hits like “I Want You Back” and “ABC” showcased not just a remarkable vocal instrument but an innate stage presence and charisma that captivated audiences. This Motown era 